In [ ]:
# !pip install langchain
# !pip install langchain-community
# !pip install -U langchain-google-genai
# !pip install --upgrade langchain-core langchain-google-vertexai
# !pip install --upgrade langchain-community duckduckgo-search
# !pip install langgraph
# !pip install requests pubchempy rdkit-pypi
# !pip install ctx-python
# !pip install biopython
# !pip install langgraph-checkpoint-sqlite
# !pip uninstall -y faiss-cpu
# !pip install langchain_openai python-dotenv
# !pip install "numpy<2"  # Use NumPy 1.x for compatibility with rdkit-pypi
# !pip install beautifulsoup4

In [1]:
import os
from typing import Optional

from dotenv import load_dotenv
from langchain_core.utils.utils import secret_from_env
from langchain_openai import ChatOpenAI
from pydantic import Field, SecretStr


load_dotenv()

class ChatOpenRouter(ChatOpenAI):
    openai_api_key: Optional[SecretStr] = Field(
        alias="api_key", default_factory=secret_from_env("OPENROUTER_API_KEY", default=None)
    )
    @property
    def lc_secrets(self) -> dict[str, str]:
        return {"openai_api_key": "OPENROUTER_API_KEY"}

    def __init__(self,
                 openai_api_key: Optional[str] = None,
                 **kwargs):
        openai_api_key = openai_api_key or os.environ.get("OPENROUTER_API_KEY")
        super().__init__(base_url="https://openrouter.ai/api/v1", openai_api_key=openai_api_key, **kwargs)



/home/chen/proj/db_form_agent_system/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -------------------- stdlib --------------------
from pathlib import Path
import re, logging, warnings, sqlite3, os, time, random, pickle, requests
import json
import copy
from typing import List, TypedDict

# -------------------- langchain / langgraph --------------------
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import create_react_agent
from langgraph.errors import GraphRecursionError
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# -------------------- local tools --------------------
import sys
sys.path.append('../code')
from compound_tool import CompoundDatabaseTool
from searching_tool import FullTextJournalSearchTool
from fulltext_vector_tools import FullTextIndexTool, FullTextQueryTool

# -------------------- logging --------------------
logging.basicConfig(level=logging.WARNING, format='%(levelname)s | %(name)s | %(message)s')
logging.getLogger('httpx').setLevel(logging.WARNING)

# -------------------- constants --------------------
MAX_ITERATIONS = 20

def _vprint(*args, **kwargs):
    # set NODE_LOGS=1 to enable; anything else disables
    if os.getenv("NODE_LOGS", "1") in ("1", "true", "TRUE", "yes", "YES"):
        print(*args, **kwargs)

# -------------------- prompts --------------------
PROMPTS_DIR = Path("/home/chen/proj/db_form_agent_system/prompts")
LITERATURE_PROMPT_PATH = PROMPTS_DIR / "literature_agent.md"
CRITIQUE_PROMPT_PATH   = PROMPTS_DIR / "critique_agent.md"

def _load_prompt(path: Path, fallback: str) -> SystemMessage:
    try:
        return SystemMessage(content=path.read_text(encoding="utf-8"))
    except Exception:
        return SystemMessage(content=fallback.strip())

LITERATURE_AGENT_PROMPT = _load_prompt(
    LITERATURE_PROMPT_PATH,
    fallback="""
You are a meticulous research agent. Use tools when needed. Return JSON with a top-level "synthesis" object
including keys: claim (string), confidence_score (0..1), explanation (string), limitations (string), supporting_items (list).
""",
)

CRITIQUE_AGENT_PROMPT = _load_prompt(
    CRITIQUE_PROMPT_PATH,
    fallback="""
You are a critical evaluator. Read the researcher's output and decide if we should stop. Return JSON with keys:
decision ("approve" | "clarify" | "reject"), stop_reason (APPROVED | NO_NEW_EVIDENCE | STAGNANT_CLAIM | TOOL_UNAVAILABLE | FUTILITY_DETECTED | ""), 
supporting_sources (list), tool_name (string), agreement_score (0..1).
""",
)

# -------------------- model helpers --------------------
def _init_models():
    """
    Initialize research & evaluator chat models from environment variables:
      RESEARCH_MODEL    (default: 'gemini-2.5-flash')
      EVALUATOR_MODEL   (default: 'deepseek/deepseek-chat-v3.1:free')
      OPENAI_API_KEY etc. are read by the underlying client(s).
    """
    research_model = ChatOpenRouter(model_name="google/gemini-2.5-flash", temperature=0.0)

    evaluator_model = ChatOpenRouter(model_name="deepseek/deepseek-chat-v3.1:free", temperature=0.0)
    return research_model, evaluator_model

# -------------------- tool factory --------------------
def _build_tools(results_base_dir: str) -> List:
    """
    Build your tools with explicit configuration.
    Toggle sources via env:
      INCLUDE_PUBMED=1
      INCLUDE_PREPRINTS=1
      INCLUDE_CROSSREF=1
      INCLUDE_BIORXIV=1
    """
    include_pubmed     = os.getenv("INCLUDE_PUBMED", "1") in ("1","true","TRUE","yes","YES")
    include_preprints  = os.getenv("INCLUDE_PREPRINTS", "1") in ("1","true","TRUE","yes","YES")
    include_crossref   = os.getenv("INCLUDE_CROSSREF", "1") in ("1","true","TRUE","yes","YES")
    include_biorxiv    = os.getenv("INCLUDE_BIORXIV", "1") in ("1","true","TRUE","yes","YES")

    ftjs = FullTextJournalSearchTool(
        results_base_dir=results_base_dir,
        include_pubmed=include_pubmed,
        include_preprints=include_preprints,
        include_crossref=include_crossref,
        include_biorxiv=include_biorxiv,
    )

    tool_list = [ftjs, FullTextIndexTool(), FullTextQueryTool()]
    if CompoundDatabaseTool is not None:
        tool_list.insert(0, CompoundDatabaseTool())
    return tool_list

# -------------------- build agents --------------------
memory = MemorySaver()
research_model, evaluator_model = _init_models()
tools = _build_tools("/home/chen/proj/db_form_agent_system/searching_results")

agent_executor = create_react_agent(
    research_model, tools, prompt=LITERATURE_AGENT_PROMPT, checkpointer=memory
)
critique_agent_executor = create_react_agent(
    evaluator_model, tools, prompt=CRITIQUE_AGENT_PROMPT, checkpointer=memory
)

# -------------------- state definitions --------------------
class StateHistory(TypedDict):
    claim: str
    confidence_score: float
    supporting_sources: List[str]
    tool_name: str | None
    stop_reason: str | None

class AgentState(TypedDict):
    messages: List[HumanMessage | AIMessage]
    original_query: str
    stop_history: List[StateHistory]
    consecutive_no_new_evidence: int
    consecutive_stagnant_claim: int
    consecutive_tool_unavailable: int
    last_critique_signature: str | None
    consecutive_clarify_without_progress: int

# -------------------- nodes --------------------
def researcher_node(state: AgentState):
    _vprint("-----Researcher Node Invoked-----")
    last = state["messages"][-1].content if state.get("messages") else ""
    next_query = None

    m = re.search(r"\{[\s\S]*\}", str(last))
    if m:
        try:
            obj = json.loads(m.group(0))
            if (obj.get("decision","") or "").lower() == "clarify":
                next_query = obj.get("next_query")
        except json.JSONDecodeError:
            pass
    if not next_query and "clarify" in str(last).lower():
        try:
            obj = json.loads(m.group(0)) if m else {}
            next_query = obj.get("next_query") or None
        except Exception:
            pass

    if next_query:
        merged = (
            f"Original question:\n{state.get('original_query','')}\n\n"
            f"Follow-up request from reviewer:\n{next_query}\n\n"
            "Answer the follow-up strictly in the context of the original question."
        )
        input_messages = [HumanMessage(content=merged)]
    else:
        input_messages = state["messages"]

    result = agent_executor.invoke({"messages": input_messages})
    return {"messages": state["messages"] + [AIMessage(content=result['messages'][-1].content)]}

def critique_node(state: AgentState):
    _vprint("-----Critique Node Invoked-----")
    last_message = state["messages"][-1].content
    original = state.get("original_query", "")
    critique_input = f"Original question: {original}\nResearcher output:\n{last_message}"
    critique_result = critique_agent_executor.invoke({"messages": [HumanMessage(content=critique_input)]})
    critique_content = critique_result['messages'][-1].content
    raw = (critique_content or "").strip()

    parsed: dict | None = None
    try:
        candidate = json.loads(raw)
        if isinstance(candidate, dict): parsed = candidate
    except Exception:
        pass
    if parsed is None:
        match = re.search(r"\{[\s\S]*?\}", raw)
        if match:
            try:
                candidate = json.loads(match.group(0))
                if isinstance(candidate, dict): parsed = candidate
            except Exception:
                parsed = None

    stop_reason = None
    supporting_sources: List[str] = []
    tool_name: str | None = None

    if parsed:
        candidate_reason = parsed.get("stop_reason")
        if isinstance(candidate_reason, str) and candidate_reason.strip():
            stop_reason = candidate_reason.strip().upper()
        sources = parsed.get("supporting_sources")
        if isinstance(sources, list):
            supporting_sources = [str(item) for item in sources]
        candidate_tool = parsed.get("tool_name")
        if isinstance(candidate_tool, str) and candidate_tool.strip():
            tool_name = candidate_tool.strip()

    def _last_researcher_json(messages):
        if not messages:
            return ""
        for msg in reversed(messages):
            content = getattr(msg, "content", "") or ""
            if '"synthesis"' in content:
                return content
        return ""

    def _safe_float(value):
        try:
            if isinstance(value, (int, float)):
                return float(value)
            if isinstance(value, str):
                return float(value.replace("–", "-").strip())
        except Exception:
            return None
        return None

    def _extract_claim_and_confidence(messages):
        blob = _last_researcher_json(messages)
        claim = "Unclear"
        confidence = 0.0
        if not blob:
            return claim, confidence
        try:
            obj = json.loads(blob)
            synthesis = obj.get("synthesis", {}) if isinstance(obj, dict) else {}
            claim = (synthesis.get("claim") or "").strip().capitalize() if isinstance(synthesis, dict) else ""
            candidate_conf = _safe_float(synthesis.get("confidence_score")) if isinstance(synthesis, dict) else None
            if candidate_conf is not None:
                confidence = max(0.0, min(1.0, candidate_conf))
        except Exception:
            pass
        if claim == "Unclear":
            match_claim = re.search(r'"claim"\s*:\s*"([^"]+)"', blob)
            if match_claim:
                claim = match_claim.group(1).strip().capitalize()
        if confidence == 0.0:
            match_conf = re.search(r'"confidence_score"\s*:\s*("?)([0-9]*\.?[0-9]+)\1', blob)
            if match_conf:
                candidate = _safe_float(match_conf.group(2))
                if candidate is not None:
                    confidence = max(0.0, min(1.0, candidate))
        return claim, confidence

    claim, confidence = _extract_claim_and_confidence(state.get("messages", []))

    history = copy.deepcopy(state.get("stop_history", []))
    history.append({
        "claim": claim,
        "confidence_score": confidence,
        "supporting_sources": supporting_sources,
        "tool_name": tool_name,
        "stop_reason": stop_reason,
    })

    prev_no_new = state.get("consecutive_no_new_evidence", 0)
    prev_stagnant = state.get("consecutive_stagnant_claim", 0)
    prev_tool_unavailable = state.get("consecutive_tool_unavailable", 0)
    clarify_without_progress = state.get("consecutive_clarify_without_progress", 0)
    previous_signature = state.get("last_critique_signature") or ""

    raw_signature = json.dumps(parsed, sort_keys=True) if parsed else raw

    no_new = prev_no_new
    stagnant = prev_stagnant
    tool_unavailable = prev_tool_unavailable

    decision = parsed.get("decision") if parsed else None
    if isinstance(decision, str) and decision.lower() == "clarify" and not stop_reason:
        if raw_signature == previous_signature:
            clarify_without_progress += 1
        else:
            clarify_without_progress = 1
        if clarify_without_progress >= 3:
            stop_reason = "STAGNANT_CLAIM"
            stagnant = max(stagnant, clarify_without_progress)
    else:
        clarify_without_progress = 0

    if not stop_reason and (isinstance(decision, str) and decision.lower() == "clarify") and clarify_without_progress >= 2:
        stop_reason = "NO_NEW_EVIDENCE"
        history[-1]["stop_reason"] = stop_reason
        no_new = prev_no_new + 1

    if history:
        history[-1]["stop_reason"] = stop_reason

    _vprint(f"[critique_node] stop_reason={stop_reason}, clarify_without_progress={clarify_without_progress}")

    patched_content = critique_content
    if parsed is not None:
        patched = dict(parsed)
        if stop_reason:
            patched["stop_reason"] = stop_reason
        patched_content = json.dumps(patched, indent=2)

    _vprint(f"[critique_node] stop_reason={stop_reason}, clarify_without_progress={clarify_without_progress}")

    return {
        "messages": state["messages"] + [AIMessage(content=patched_content)],
        "stop_history": history,
        "consecutive_no_new_evidence": no_new,
        "consecutive_stagnant_claim": stagnant,
        "consecutive_tool_unavailable": prev_tool_unavailable,
        "last_critique_signature": raw_signature,
        "consecutive_clarify_without_progress": clarify_without_progress,
    }

def should_continue(state: AgentState):
    """Decide whether to continue the loop using critique stop reasons."""
    last_message_content = state["messages"][-1].content if state.get("messages") else ""
    raw = (last_message_content or "").strip()
    _vprint(f"[should_continue] raw critique output: {raw}...")

    STOP_REASONS = {"APPROVED", "NO_NEW_EVIDENCE", "STAGNANT_CLAIM", "TOOL_UNAVAILABLE", "FUTILITY_DETECTED"}

    def _normalized_reason(value) -> str | None:
        if isinstance(value, str) and value.strip():
            return value.strip().upper()
        return None

    history = state.get("stop_history") or []
    if history:
        latest_reason = _normalized_reason(history[-1].get("stop_reason"))
        if latest_reason in STOP_REASONS:
            _vprint(f"-----Agent Decided to Stop (history stop_reason={latest_reason})-----")
            return END

    if state.get("consecutive_clarify_without_progress", 0) >= 5:
        _vprint("-----Agent Decided to Stop (clarify stagnation >=5)-----")
        return END

    def _should_stop_from_json(obj: dict) -> bool:
        stop_reason = _normalized_reason(obj.get("stop_reason"))
        if stop_reason in STOP_REASONS:
            _vprint(f"-----Agent Decided to Stop (stop_reason={stop_reason})-----")
            return True

        decision = obj.get("decision")
        if isinstance(decision, str) and decision.lower() == "approve":
            _vprint("-----Agent Decided to Stop (legacy decision=approve)-----")
            return True

        try:
            score = float(obj.get("agreement_score", 0.0))
            if score > 0.75:
                _vprint(f"-----Agent Decided to Stop (legacy agreement_score={score:.2f} > 0.75)-----")
                return True
        except Exception:
            pass

        return False

    try:
        critique_json = json.loads(raw)
        if isinstance(critique_json, dict) and _should_stop_from_json(critique_json):
            return END
    except Exception:
        pass

    match = re.search(r"\{[\s\S]*?\}", raw)
    if match:
        try:
            critique_json = json.loads(match.group(0))
            if isinstance(critique_json, dict) and _should_stop_from_json(critique_json):
                _vprint("-----Agent Decided to Stop (extracted JSON)-----")
                return END
        except Exception:
            pass

    low = raw.lower()
    if any(token in low for token in ["approve", "approved"]):
        _vprint("-----Agent Decided to Stop (text match)-----")
        return END

    return "researcher"

# -------------------- workflow --------------------
workflow = StateGraph(AgentState)
workflow.add_node("researcher", researcher_node)
workflow.add_node("critique", critique_node)
workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "critique")
workflow.add_conditional_edges("critique", should_continue)

app = workflow.compile(checkpointer=memory)

# -------------------- helpers --------------------
def _last_researcher_json(messages):
    """Return the last message content that contains a 'synthesis' JSON, else ''."""
    if not messages:
        return ""
    for msg in reversed(messages):
        content = getattr(msg, "content", "") or ""
        if '"synthesis"' in content:
            return content
    return ""

def determine_final_claim(messages):
    """Read 'synthesis.claim' with minimal normalization."""
    blob = _last_researcher_json(messages)
    if not blob:
        return "Unclear"
    try:
        obj = json.loads(blob)
        syn = obj.get("synthesis", {}) if isinstance(obj, dict) else {}
        claim_raw = syn.get("claim")
        if isinstance(claim_raw, str) and claim_raw.strip():
            return claim_raw.strip()
    except Exception:
        pass

    m = re.search(r'"claim"\s*:\s*"([^"]+)"', blob)
    if m:
        candidate = m.group(1).strip()
        if candidate:
            return candidate
    return "Unclear"

def _safe_float(x):
    try:
        if isinstance(x, (int, float)):
            return float(x)
        if isinstance(x, str):
            return float(x.replace("–", "-").strip())
    except Exception:
        return None

def determine_confidence_score(messages):
    """
    Reads numeric 'synthesis.confidence_score' (0..1).
    If missing/unparseable, returns 0.0 (so it won't cross 0.45).
    """
    blob = _last_researcher_json(messages)
    if not blob:
        return 0.0
    try:
        obj = json.loads(blob)
        syn = obj.get("synthesis", {}) if isinstance(obj, dict) else {}
        score = syn.get("confidence_score")
        return _safe_float(score)
    except Exception:
        pass
    m = re.search(r'"confidence_score"\s*:\s*([0-9.]+)', blob)
    if m:
        try:
            return float(m.group(1))
        except Exception:
            pass
    return 0.0


In [3]:
def main():
    _vprint("\n🧪 Analytical Chemistry Agent Ready!")
    while True:
        try:
            query = input("\n📝 Enter your question (or 'quit' to exit): ")
        except (EOFError, KeyboardInterrupt):
            _vprint("\nExiting.")
            break

        _vprint(f"\n>>> User input query: {query}\n", flush=True)
        if query.lower().strip() == 'quit':
            break

        # per-run session dir for all tools to reuse
        query_timestamp = time.strftime("%Y%m%d_%H%M%S", time.gmtime())
        query_results_dir = f"/home/chen/proj/db_form_agent_system/searching_results/run_{query_timestamp}"
        os.environ["SEARCH_SESSION_DIR"] = query_results_dir

        # optional tracing/project metadata
        os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "true")
        os.environ["LANGCHAIN_PROJECT"] = f"DB Form Agent Run - {query_timestamp}"
        _vprint(f"📁 All results will be saved to: searching_results/run_{query_timestamp}\n")

        # allow users to force label set with:  answer: "Endogenous" "Exogenous"
        requested_labels: list[str] = []
        directive = query
        match = re.search(r"(?i)answer:\s*(.*)", query)
        if match:
            raw_segment = match.group(1)
            candidates = [label.strip() for label in re.findall(r'"([^"]+)"', raw_segment)]
            for label in candidates:
                if label and label not in requested_labels:
                    requested_labels.append(label)
            if requested_labels:
                label_text = ", ".join(f'"{label}"' for label in requested_labels)
                directive = (
                    f"{query}\n\nInstruction: ensure synthesis.claim exactly matches one of the user-specified answer options: {label_text}. "
                    "Select the option best supported by the evidence."
                )

        init_message = HumanMessage(content=directive)
        init_state = {
            'messages': [init_message],
            'original_query': query,
            'stop_history': [],
            'consecutive_no_new_evidence': 0,
            'consecutive_stagnant_claim': 0,
            'consecutive_tool_unavailable': 0,
            'last_critique_signature': None,
            'consecutive_clarify_without_progress': 0,
        }

        thread_id = f"qa_session_{int(time.time())}"
        last_event_state = None

        try:
            for event in app.stream(
                init_state,
                config={'configurable': {'thread_id': thread_id}, 'recursion_limit': 20}
            ):
                for v in event.values():
                    last_event_state = v
                    try:
                        content = v['messages'][-1].content
                        try:
                            obj = json.loads(content)
                            _vprint(json.dumps(obj, indent=2))
                        except json.JSONDecodeError:
                            _vprint(content)
                    except Exception:
                        _vprint(str(v))
        except GraphRecursionError as ge:
            _vprint(f"[GraphRecursionError] {ge}")
        except KeyboardInterrupt:
            _vprint("\nInterrupted by user.")
            break
        except Exception as e:
            _vprint(f"[ERROR] Agent execution failed: {e}")

        claim = 'Unclear'
        score = 0.0
        critique_decision = None
        critique_json = {}

        if last_event_state and last_event_state.get('messages'):
            claim = determine_final_claim(last_event_state['messages'])
            score = determine_confidence_score(last_event_state['messages']) or 0.0

        display_label = claim.strip() if isinstance(claim, str) and claim.strip() else "Unclear"

        if last_event_state and last_event_state.get('messages'):
            critique_msg = last_event_state['messages'][-1]
            critique_raw = getattr(critique_msg, 'content', '') or ''
            try:
                critique_json = json.loads(critique_raw)
            except Exception:
                match_json = re.search(r"\{[\s\S]*?\}", critique_raw)
                if match_json:
                    try:
                        critique_json = json.loads(match_json.group(0))
                    except Exception:
                        critique_json = {}
            if isinstance(critique_json, dict):
                decision_val = critique_json.get('decision')
                if isinstance(decision_val, str):
                    critique_decision = decision_val.strip().lower()

        _vprint(f"\n=== FINAL DECISION: {display_label} (confidence {score:.2f}) ===")
        if critique_decision:
            _vprint(f"    critique decision: {critique_decision}")
        _vprint(f"    researcher claim: {claim}")

if __name__ == '__main__':
    main()


🧪 Analytical Chemistry Agent Ready!



>>> User input query: is TG(i-16:0/a-17:0/18:0)[rac] endogenous metabolite from human? Answer: "Endogenous" or "Exogenous"

📁 All results will be saved to: searching_results/run_20251020_184259

-----Researcher Node Invoked-----


2025-10-20 18:43:03,921 | INFO | Reusing existing session directory: /home/chen/proj/db_form_agent_system/searching_results/run_20251020_184259
2025-10-20 18:43:03,924 | INFO | Starting search #1/3 for query: 'TG(i-16:0/a-17:0/18:0)[rac] human endogenous'; saving to /home/chen/proj/db_form_agent_system/searching_results/run_20251020_184259
2025-10-20 18:43:03,926 | INFO | [1/3] Searching PubMed/PMC…
2025-10-20 18:43:04,356 | INFO | [1/3] Searching Crossref…
2025-10-20 18:43:05,030 | INFO | [1/3] Searching arXiv…
2025-10-20 18:43:05,441 | INFO | [1/3] Searching bioRxiv…
2025-10-20 18:43:10,139 | INFO | Starting search #2/3 for query: 'PubChem CID 131812506 human metabolite'; saving to /home/chen/proj/db_form_agent_system/searching_results/run_20251020_184259
2025-10-20 18:43:10,141 | INFO | [2/3] Searching PubMed/PMC…
2025-10-20 18:44:09,854 | INFO | [2/3] Searching Crossref…
2025-10-20 18:44:10,499 | INFO | [2/3] Searching arXiv…
2025-10-20 18:44:10,776 | INFO | [2/3] Searching bioRxiv

THOUGHT: Despite multiple attempts with various search queries, including the specific compound name, its PubChem CID, its full chemical name, and its constituent fatty acids, no direct evidence has been found to suggest that TG(i-16:0/a-17:0/18:0)[rac] is an endogenous human metabolite. The search results consistently returned null, indicating a lack of literature on this specific triglyceride in the context of human endogenous metabolism.

Therefore, based on the absence of evidence, the most reasonable conclusion is that it is not an endogenous human metabolite. I will classify it as "Exogenous" and provide a low confidence score due to the lack of direct evidence.
```json
{
  "synthesis": {
    "claim": "Exogenous",
    "explanation": "Despite multiple comprehensive literature searches using various specific and broad terms related to TG(i-16:0/a-17:0/18:0)[rac], its chemical name, PubChem CID, and constituent fatty acids in the context of human metabolism, no evidence was found to

In [ ]:
# is TG(i-16:0/a-17:0/18:0)[rac] endogenous metabolite from human? Answer: "Endogenous" or "Exogenous"

In [ ]:
import pandas as pd
from tqdm import tqdm


def is_endogenous_compound(compound_name: str) -> str:
    prompt = f"Is the compound '{compound_name}' an endogenous metabolite from human? Answer 'Endogenous', 'Exogenous'."

    query_timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
    query_results_dir = f"/home/chen/proj/db_form_agent_system/searching_results/run_{query_timestamp}"
    os.environ["SEARCH_SESSION_DIR"] = query_results_dir

    requested_labels: list[str] = []
    match = re.search(r"(?i)answer:\s*(.*)", prompt)
    if match:
        raw_segment = match.group(1)
        candidates = [label.strip() for label in re.findall(r'"([^\"]+)"', raw_segment)]
        for label in candidates:
            if label and label not in requested_labels:
                requested_labels.append(label)
        if requested_labels:
            label_text = ", ".join(f'"{label}"' for label in requested_labels)
            prompt = (
                f"{prompt}\n\nInstruction: ensure synthesis.claim exactly matches one of the user-specified answer options: {label_text}. "
                "Select the option best supported by the evidence."
            )

    init_message = HumanMessage(content=prompt)
    init_state = {
        'messages': [init_message],
        'original_query': prompt,
        'stop_history': [],
        'consecutive_no_new_evidence': 0,
        'consecutive_stagnant_claim': 0,
        'consecutive_tool_unavailable': 0,
        'last_critique_signature': None,
        'consecutive_clarify_without_progress': 0,
    }

    thread_id = f"qa_session_{int(time.time())}"
    last_event_state = None
    try:
        for event in app.stream(init_state, config={'configurable': {'thread_id': thread_id}, 'recursion_limit': 20}):
            for v in event.values():
                last_event_state = v
                try:
                    _vprint(v['messages'][-1].content)
                except Exception:
                    _vprint(str(v))
    except GraphRecursionError:
        _vprint("-----Reached max iteration limit; stopping further processing-----")
        return "Reached max iteration"

    claim = 'Unclear'

    if last_event_state and last_event_state.get('messages'):
        claim = determine_final_claim(last_event_state['messages'])

    return claim


df = pd.read_csv("/data/chen/data/hmdb/metabolites.csv")

# get 100 rows with 80% endogenous
df_endogenous = df[df['Endogenous']=='Yes'].sample(n=45, random_state=42)
df_non_endogenous = df[df['Endogenous']=='No'].sample(n=5, random_state=42)
df_sampled = pd.concat([df_endogenous, df_non_endogenous]).reset_index(drop=True)

tqdm.pandas(desc="Checking compounds") 
df_sampled['Endogenous_LLM'] = df_sampled['Name'].progress_apply(is_endogenous_compound)
df_sampled[['Name', 'Endogenous', 'Endogenous_LLM']]


In [ ]:
df_sampled[['Name', 'Endogenous', 'Endogenous_LLM']]

In [ ]:
df_sampled

In [ ]:
import pandas as pd
df = pd.read_csv('/data/chen/data/hmdb/metabolites_endo_exo.csv')

df['Endogenous_LLM'] = df['Name'].apply(is_endogenous_compound)
df[['Name', 'Origin', 'Endogenous_LLM']]